# ⭐ Aula 16 — Bons e maus estimadores nas avaliações de um app

Este notebook **não tem nota nem entrega**, e você **não precisa saber programar**. É só executar as células, na ordem, de cima para baixo, e ler os textos entre elas.

Neste notebook você faz parte do **time de dados de uma loja de aplicativos**. Os usuários dão notas de 1 a 5 estrelas aos apps, e o time usa amostras dessas notas para estimar a média e a variância das notas. A Aula 16 traz as **qualidades de um bom estimador**: **ausência de viés**, **consistência** e **eficiência**.

> ⚠️ Os números usados aqui são **fictícios**, escolhidos só para ilustrar os conceitos.

Cada seção segue sempre a mesma ordem:
1. **A situação e a pergunta** (o enunciado, como num exercício);
2. **A ferramenta da aula** que responde a pergunta;
3. **A resposta pela conta**, quando há conta;
4. **O que o código faz**, antes de cada célula;
5. **Como ler o resultado**.

> Notebook elaborado com auxílio de IA (Claude). Confira antes de usar.

## Como executar

- Cada **célula de código** (as caixas cinza) tem um botão **▶** do lado esquerdo. Clique nele, ou clique na célula e aperte **Shift + Enter**.
- Execute **uma de cada vez, na ordem**: algumas células usam o resultado das anteriores.
- Na primeira vez, o Colab pode avisar que o notebook "não foi criado pelo Google". Clique em **Executar mesmo assim**.
- Se preferir, use o menu **Ambiente de execução → Executar tudo** e depois só leia, de cima para baixo.
- Detalhe: o Python escreve números com **ponto** no lugar da vírgula (`0.26` = 0,26).

## 0. Preparando as ferramentas

A célula abaixo carrega:

- **`numpy`**: uma "calculadora" que faz contas com milhares de números de uma vez e sabe **sortear** números aleatórios;
- **`matplotlib`**: desenha **gráficos**.

Criamos também o **sorteador** (`rng`) com a **semente 42**, que faz as simulações saírem sempre iguais.

**Lembrete:** **empírico** = o que sai da simulação; **teórico** = o que a fórmula da aula diz.

In [ ]:
import numpy as np                    # calculadora de números e sorteios
import matplotlib.pyplot as plt       # gráficos

rng = np.random.default_rng(42)       # sorteador, semente 42
print("Tudo pronto!")

## 1. As notas de um app

As notas de um app famoso seguem um padrão comum em lojas de aplicativos: quem gosta dá **5 estrelas**, quem teve problema dá **1 estrela**, e quase ninguém fica no meio. A distribuição das notas na população é:

| nota | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|
| probabilidade | 0,25 | 0,05 | 0,05 | 0,15 | 0,50 |

Com isso, $\mu = 3{,}6$ estrelas e $\sigma^2 = 2{,}84$. Na vida real o time **não conhece** esses valores: é justamente o que quer estimar. Aqui fingimos conhecer, para poder **testar** os estimadores.

### O que o código faz
Guarda as notas e as probabilidades e calcula $\mu$ e $\sigma^2$ pelas fórmulas de variável aleatória discreta (Aula 7).

In [ ]:
notas = np.array([1, 2, 3, 4, 5])
probs = np.array([0.25, 0.05, 0.05, 0.15, 0.50])

mu = (notas * probs).sum()                       # E(X)
sigma2 = ((notas - mu)**2 * probs).sum()         # Var(X)
print(f"μ = {mu:.2f}   σ² = {sigma2:.2f}")

## 2. Ausência de viés: dividir a variância por $n$ ou por $n - 1$?

### A situação e a pergunta
Para estimar a variância das notas com amostras de **5 avaliações**, alguém do time usou a fórmula "natural"
$$\hat\sigma^2 = \frac{1}{n}\sum_{i=1}^{n} (X_i - \bar X)^2,$$
dividindo por $n$, como na Aula 2. Outra pessoa usou a **variância amostral** $S^2$, que divide por $n - 1$. **Qual das duas acerta, em média?**

### A ferramenta da aula
Um estimador é **não viesado** se $E(\hat\theta) = \theta$: em média, sobre todas as amostras, ele acerta o parâmetro. Na aula vimos (sem demonstração) que
$$E(\hat\sigma^2) = \sigma^2\left(1 - \frac{1}{n}\right) \qquad \text{e} \qquad E(S^2) = \sigma^2 .$$

### A resposta pela conta
Com $n = 5$: $E(\hat\sigma^2) = 2{,}84 \times \left(1 - \frac{1}{5}\right) = 2{,}84 \times 0{,}8 = 2{,}272$, enquanto $E(S^2) = 2{,}84$.

### O que o código faz
Sorteia **100.000 amostras de 5 notas** e calcula os dois estimadores em cada uma. No `numpy`, a opção `ddof` escolhe o divisor: `ddof=0` divide por $n$ e `ddof=1` divide por $n - 1$. Depois tira a **média** de cada estimador sobre as 100.000 amostras (a média das estimativas faz o papel de $E(\hat\theta)$).

In [ ]:
amostras = rng.choice(notas, size=(100_000, 5), p=probs)       # 100.000 amostras de 5 notas

sigma2_chapeu = amostras.var(axis=1, ddof=0)                   # divide por n
s2 = amostras.var(axis=1, ddof=1)                              # divide por n - 1

print(f"Variância verdadeira σ² = {sigma2:.3f}")
print(f"Média de σ̂² (divide por n):     {sigma2_chapeu.mean():.3f}   (fórmula: σ²(1 - 1/n) = {sigma2 * 0.8:.3f})")
print(f"Média de S² (divide por n - 1): {s2.mean():.3f}   (fórmula: σ² = {sigma2:.3f})")

**Como ler:** dividindo por $n$, a média das estimativas fica em cerca de **2,27**, abaixo do valor verdadeiro 2,84: o estimador erra **sistematicamente para baixo** (é **viesado**). Dividindo por $n - 1$, a média das estimativas fica em cerca de **2,84**: $S^2$ é **não viesado**.

É por isso que calculadoras, planilhas e programas de estatística usam $n - 1$ na variância amostral.

## 3. Consistência: mais avaliações, estimativa mais perto do parâmetro

### A situação e a pergunta
O time acompanha a nota média de um app **novo** à medida que as avaliações chegam. A média verdadeira (que o time não conhece) é $\mu = 3{,}6$. Duas regras são comparadas:
- **a média de todas as avaliações recebidas até o momento** ($\bar x$);
- **só a primeira avaliação** recebida.

**O que acontece com cada estimativa conforme o número de avaliações cresce?**

### A ferramenta da aula
Um estimador é **consistente** quando, à medida que $n$ cresce, $E(\hat\theta)$ se aproxima de $\theta$ e $\text{Var}(\hat\theta)$ se aproxima de 0: com amostras maiores, as estimativas ficam cada vez mais perto do parâmetro. Para a média, $E(\bar X) = \mu$ e $\text{Var}(\bar X) = \sigma^2/n$, que vai a 0. Já a primeira avaliação tem variância $\sigma^2$, qualquer que seja $n$.

### O que o código faz
1. sorteia **5 sequências** de 2.000 avaliações (5 "apps" com a mesma distribuição de notas);
2. para cada sequência, calcula a média **acumulada** (a média das primeiras 1, 2, 3, ..., 2.000 avaliações);
3. desenha as 5 curvas e a linha do valor verdadeiro $\mu = 3{,}6$;
4. no fim, compara as duas regras em 100.000 apps: em quantos a estimativa ficou a **no máximo 0,5 estrela** de 3,6 (entre 3,1 e 4,1).

In [ ]:
for _ in range(5):
    sequencia = rng.choice(notas, size=2000, p=probs)             # 2.000 avaliações em sequência
    media_acumulada = np.cumsum(sequencia) / np.arange(1, 2001)   # média das primeiras k notas
    plt.plot(range(1, 2001), media_acumulada, linewidth=1)

plt.axhline(mu, color="black", linestyle="--", label="μ = 3,6")
plt.xscale("log")                                              # eixo em escala log, para ver o começo
plt.xlabel("Número de avaliações (escala logarítmica)")
plt.ylabel("Nota média até aquele momento")
plt.title("Consistência: a média amostral se aproxima de μ")
plt.legend()
plt.show()

for n in [10, 100, 1000]:
    amostras = rng.choice(notas, size=(100_000, n), p=probs)
    perto_media = (np.abs(amostras.mean(axis=1) - mu) <= 0.5).mean()
    perto_primeira = (np.abs(amostras[:, 0] - mu) <= 0.5).mean()
    print(f"n = {n:4d}:  média a até 0,5 de μ em {perto_media:.0%} dos apps;  "
          f"primeira avaliação em {perto_primeira:.0%}")

**Como ler:** com poucas avaliações, as 5 curvas oscilam muito — um app pode parecer ter nota 2 ou nota 5 nas primeiras avaliações. Conforme as avaliações se acumulam, todas as curvas se **aproximam de 3,6** e as oscilações diminuem. (O eixo horizontal está em escala logarítmica: cada marca vale 10 vezes a anterior.)

Na tabela impressa, a chance de a média ficar perto de 3,6 sobe com $n$ (a média é **consistente**). A primeira avaliação só fica entre 3,1 e 4,1 quando é nota 4, o que acontece em 15% dos apps, qualquer que seja $n$: ela não melhora, pois ignora todas as outras avaliações (**não é consistente**, mesmo sendo não viesada).

**Conclusão:** é por isso que lojas de aplicativos mostram "poucas avaliações" para apps novos: com $n$ pequeno, a média ainda não é confiável.

## 4. Eficiência: média ou mediana?

### A situação e a pergunta
Num serviço de streaming, o tempo de carregamento de um vídeo é normal com média $\mu = 3$ s e desvio padrão $\sigma = 1$ s. Para estimar $\mu$ com amostras de **25** vídeos, dá para usar a **média** ou a **mediana** da amostra: como a normal é simétrica, as duas acertam em média. **Qual das duas é melhor?**

### A ferramenta da aula
Entre dois estimadores **não viesados**, o mais **eficiente** é o de **menor variância**: ele erra menos de amostra para amostra.

### O que o código faz
Sorteia **100.000 amostras de 25 tempos**, calcula a média e a mediana de cada uma e compara a média e a variância dos dois estimadores.

In [ ]:
amostras = rng.normal(3, 1, size=(100_000, 25))               # 100.000 amostras de 25 tempos

est_media = amostras.mean(axis=1)
est_mediana = np.median(amostras, axis=1)

print(f"Média:    E = {est_media.mean():.3f}   Var = {est_media.var():.4f}   (fórmula: σ²/n = {1 / 25:.4f})")
print(f"Mediana:  E = {est_mediana.mean():.3f}   Var = {est_mediana.var():.4f}")
print(f"A variância da mediana é {est_mediana.var() / est_media.var():.2f} vezes a da média")

**Como ler:** os dois estimadores acertam em média (as duas médias ficam em 3,00), mas a **variância da mediana é cerca de 1,5 vez** a da média. Então, para dados normais, a média é **mais eficiente**: com o mesmo número de vídeos, ela dá estimativas mais concentradas. Para a mediana ter a mesma precisão, seria preciso uma amostra cerca de 50% a 60% maior (na aula: cerca de 40 pedidos contra 25).

(A mediana tem outras vantagens: se a amostra tiver valores absurdos — um vídeo que travou e "demorou" 10 minutos —, ela é muito menos afetada que a média.)

## Para fechar

Neste notebook, o time de dados usou as ferramentas da Aula 16:

- **ausência de viés:** dividir a variância por $n$ subestima $\sigma^2$; dividir por $n - 1$ ($S^2$) acerta em média;
- **consistência:** a média amostral se aproxima de $\mu$ conforme $n$ cresce; a "primeira avaliação" é não viesada, mas não melhora;
- **eficiência:** para dados normais, a média é mais precisa que a mediana.

Na **Aula 17** vêm duas distribuições novas, a **t de Student** e a **qui-quadrado**, que aparecem quando precisamos **estimar $\sigma$** a partir da amostra — com o $S$ desta aula.